# Domain-adversarial multi-species Mask R-CNN

This notebook is a DA-enabled revision of the mixed-species Mask R-CNN
training workflow. It retains the original COCO data, augmentation,
optimizer, patch-size, and evaluation workflow where possible, while
adding an explicit species-domain classifier and gradient reversal.


reference : KChicken Detection/Segmentation - MaskRCNN Pytorch
https://www.kaggle.com/code/hayriyigit/chicken-detection-segmentation-maskrcnn-pytorch

- understood and implemented 0ct 14 JS

another reference:
- Model: Mask R-CNN, Kaiming He and Georgia Gkioxari and Piotr Dollár and Ross Girshick,2018, arXiv,cs.CV,https://arxiv.org/abs/1703.06870
- Model: Benchmarking detection transfer learning with vision transformers. Y Li, S Xie, X Chen, P Dollar, K He, R Girshick
arXiv preprint arXiv:2111.11429, 2021

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import seaborn as sns


In [ ]:
# --- Reproducible DA configuration ---
GPU_ID = 0
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

image_size = 256  # 256 or 512; run the two conditions separately.
NUM_CLASSES = 5   # background + vessel + ray + fibre + parenchyma

# The mapping is fixed and must be retained in every checkpoint/log.
DOMAIN_CODE_TO_ID = {
    "acapod": 0,  # Acacia podalyriifolia
    "alllit": 1,  # Allocasuarina littoralis
    "burspi": 2,  # Bursaria spinosa
    "euclon": 3,  # Eucalyptus longifolia
    "exostr": 4,  # Exocarpos strictus
    "hakdac": 5,  # Hakea dactyloides
}
DOMAIN_ID_TO_CODE = {value: key for key, value in DOMAIN_CODE_TO_ID.items()}
NUM_DOMAINS = len(DOMAIN_CODE_TO_ID)

# Fix these before full training. Changing either value requires a new run.
GRL_COEFFICIENT = 1.0
DOMAIN_LOSS_WEIGHT = 0.10

LEARNING_RATE = 1e-5
WEIGHT_DECAY = 5e-4
BATCH_SIZE = 8
NUM_EPOCHS = 700
NUM_WORKERS = 4
PIN_MEMORY = True
SAVE_MODEL = True
LOAD_MODEL = False
RANDOM_SEED = 20260919

# The smoke test performs forward/backward checks but does not update weights.
RUN_SMOKE_TEST = True
RUN_FULL_TRAINING = False
SAVE_PROGRESS_GIF = False

TRAIN_DIR = f"dataset/png_{image_size}/image/train"
VALID_DIR = f"dataset/png_{image_size}/image/valid"
TEST_DIR = f"dataset/png_{image_size}/image/test"
IMAGE_SIZE = [image_size, image_size]

run_tag = f"{image_size}_multispecies_DA_lambda{DOMAIN_LOSS_WEIGHT:g}".replace(".", "p")
CHECKPOINT_DIR = Path("./checkpoint") / run_tag
OUTPUT_DIR = Path("./output") / run_tag
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = CHECKPOINT_DIR / "best_model.tar"

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_SEED)
print(f"Using device: {DEVICE}")
print(f"DA run: {run_tag}")


In [ ]:
DEVICE

In [ ]:
import os
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


Define test image

In [ ]:
from PIL import Image

testfilename = [fl for fl in os.listdir(TEST_DIR) if fl.endswith('.png')]
i = 4
img0 = Image.open(os.path.join(TEST_DIR,testfilename[i])).convert('RGB')
img0 = np.array(img0)
img = cv2.imread(os.path.join(TEST_DIR,testfilename[i]))


# DATASET

In [ ]:
# Read the pooled COCO data and return image, detection target, and domain ID.
class CCDataset(Dataset):
    def __init__(self, mode="train", augmentation=None):
        directory_by_mode = {
            "train": TRAIN_DIR,
            "valid": VALID_DIR,
            "test": TEST_DIR,
        }
        if mode not in directory_by_mode:
            raise ValueError(f"Unknown mode: {mode}")

        self.mode = mode
        self.dataset_path = directory_by_mode[mode]
        annotation_path = os.path.join(
            self.dataset_path, f"{mode}_annot_coco_{image_size}.json"
        )
        self.coco = COCO(annotation_path)
        self.image_ids = sorted(self.coco.getImgIds())
        self.augmentation = augmentation

        # Resolve once, so label mistakes fail before optimisation begins.
        self.samples = []
        for image_id in self.image_ids:
            image_info = self.coco.loadImgs([image_id])[0]
            file_name = image_info["file_name"]
            domain_id = self._domain_from_filename(file_name)
            self.samples.append(
                {
                    "image_id": image_id,
                    "file_name": file_name,
                    "domain_id": domain_id,
                    "domain_code": DOMAIN_ID_TO_CODE[domain_id],
                }
            )

    @staticmethod
    def _domain_from_filename(file_name):
        lowered = file_name.lower()
        matches = [
            domain_id
            for species_code, domain_id in DOMAIN_CODE_TO_ID.items()
            if species_code in lowered
        ]
        if len(matches) != 1:
            raise ValueError(
                "Could not assign exactly one species domain to "
                f"'{file_name}'. Check DOMAIN_CODE_TO_ID and merged filenames."
            )
        return matches[0]

    def __len__(self):
        return len(self.samples)

    def get_masks(self, image_id):
        annotation_ids = self.coco.getAnnIds(imgIds=[image_id])
        annotations = self.coco.loadAnns(annotation_ids)
        masks = [self.coco.annToMask(annotation) for annotation in annotations]
        cell_types = [annotation["category_id"] for annotation in annotations]
        return masks, cell_types

    @staticmethod
    def get_boxes(masks, cell_types):
        valid_masks, valid_types, boxes = [], [], []
        for mask, cell_type in zip(masks, cell_types):
            x, y, width, height = cv2.boundingRect(mask)
            if width > 0 and height > 0:
                boxes.append([x, y, x + width, y + height])
                valid_masks.append(mask)
                valid_types.append(cell_type)
        return np.array(boxes), np.array(valid_masks), np.array(valid_types)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = cv2.imread(os.path.join(self.dataset_path, sample["file_name"]))
        if image is None:
            raise FileNotFoundError(sample["file_name"])
        masks, cell_types = self.get_masks(sample["image_id"])

        if self.augmentation:
            augmented = self.augmentation(image=image, masks=masks)
            image, masks = augmented["image"], augmented["masks"]

        boxes, masks, cell_types = self.get_boxes(masks, cell_types)
        if len(boxes) == 0:
            return None

        image = torch.as_tensor(
            image.transpose(2, 0, 1) / 255.0, dtype=torch.float32
        )
        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(cell_types, dtype=torch.int64),
            "masks": torch.as_tensor(masks, dtype=torch.uint8),
        }
        domain_id = torch.tensor(sample["domain_id"], dtype=torch.long)
        return image, target, domain_id


In [ ]:
# Detection targets have variable instance counts; keep them as a list.
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None
    images, targets, domain_labels = zip(*batch)
    return list(images), list(targets), torch.stack(domain_labels)


def domain_count_report(dataset):
    counts = Counter(sample["domain_code"] for sample in dataset.samples)
    return {code: counts.get(code, 0) for code in DOMAIN_CODE_TO_ID}


def validate_domain_coverage(dataset, split_name):
    observed = {sample["domain_id"] for sample in dataset.samples}
    expected = set(DOMAIN_ID_TO_CODE)
    missing = expected - observed
    if missing:
        missing_names = [DOMAIN_ID_TO_CODE[idx] for idx in sorted(missing)]
        raise AssertionError(f"{split_name} is missing domain(s): {missing_names}")
    print(f"{split_name} domain counts: {domain_count_report(dataset)}")


Followings are for augumentation

In [ ]:
import albumentations as A
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(
        contrast_limit=0.2, brightness_limit=0.3, p=0.5),
    A.OneOf([
        A.ImageCompression(p=0.8),
        A.RandomGamma(p=0.8),
        A.Blur(p=0.8),
        A.Equalize(mode='cv',p=0.8)
    ], p=1.0),
    A.OneOf([
        A.ImageCompression(p=0.8),
        A.RandomGamma(p=0.8),
        A.Blur(p=0.8),
        A.Equalize(mode='cv',p=0.8),
    ], p=1.0)
])

# Domain-adaptive model

The detector remains Torchvision's `maskrcnn_resnet50_fpn_v2`. During
training, a forward hook obtains its FPN feature maps; global pooling,
a gradient-reversal layer (GRL), and a six-class domain classifier are
added as an adversarial branch. The detector's task loss and the
weighted domain loss are optimised jointly.


**Important comparison rule.** Use the same image split, augmentation,
initial learning rate, optimizer, epoch count, and performance-check
procedure as the mixed-species model. A DA checkpoint is not compatible
with the legacy inference/score notebooks unless this model definition
is imported there as well.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, inputs, coefficient):
        ctx.coefficient = coefficient
        return inputs.view_as(inputs)

    @staticmethod
    def backward(ctx, gradient_output):
        return -ctx.coefficient * gradient_output, None


class GradientReversalLayer(nn.Module):
    def __init__(self, coefficient=1.0):
        super().__init__()
        self.coefficient = float(coefficient)

    def forward(self, inputs):
        return GradientReversalFunction.apply(inputs, self.coefficient)


def pool_fpn_features(feature_maps):
    '''Average pooled levels of an FPN into one 256-D vector/image.'''
    maps = list(feature_maps.values())
    if not maps:
        raise RuntimeError("The Mask R-CNN backbone returned no FPN features.")
    pooled = [F.adaptive_avg_pool2d(feature_map, 1).flatten(1) for feature_map in maps]
    feature_dimensions = {feature.shape[1] for feature in pooled}
    if len(feature_dimensions) != 1:
        raise RuntimeError(f"FPN levels have inconsistent channels: {feature_dimensions}")
    return torch.stack(pooled, dim=0).mean(dim=0)


class DomainClassifier(nn.Module):
    def __init__(self, feature_dim, num_domains):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_domains),
        )

    def forward(self, pooled_features):
        return self.network(pooled_features)


class DomainAdaptiveMaskRCNN(nn.Module):
    '''Mask R-CNN plus an adversarial classifier on its FPN features.

    In training mode the return value is a loss dictionary. `loss_domain`
    is already weighted by DOMAIN_LOSS_WEIGHT; the unweighted domain loss
    and domain accuracy are retained only for logging, so they cannot be
    accidentally added twice to the optimisation objective.
    '''

    def __init__(
        self,
        detector,
        num_domains,
        domain_loss_weight,
        grl_coefficient,
        feature_dim=256,
    ):
        super().__init__()
        self.detector = detector
        self.domain_loss_weight = float(domain_loss_weight)
        self.grl = GradientReversalLayer(grl_coefficient)
        self.domain_classifier = DomainClassifier(feature_dim, num_domains)
        self._cached_backbone_features = None
        self.last_domain_loss_raw = None
        self.last_domain_accuracy = None
        self.last_pooled_feature_shape = None
        self._backbone_hook_handle = self.detector.backbone.register_forward_hook(
            self._cache_backbone_features
        )

    def _cache_backbone_features(self, module, inputs, output):
        if self.training:
            self._cached_backbone_features = output

    def forward(self, images, targets=None, domain_labels=None):
        if not self.training:
            # Normal inference: no labels or adversarial loss are required.
            return self.detector(images)

        if targets is None:
            raise ValueError("Detection targets are required while training.")
        if domain_labels is None:
            raise ValueError("Domain labels are required while training DA Mask R-CNN.")
        if len(images) != len(domain_labels):
            raise ValueError("Each image in a batch must have one domain label.")

        self._cached_backbone_features = None
        detection_losses = self.detector(images, targets)
        if self._cached_backbone_features is None:
            raise RuntimeError("FPN feature hook did not capture backbone output.")

        pooled_features = pool_fpn_features(self._cached_backbone_features)
        self.last_pooled_feature_shape = tuple(pooled_features.shape)
        domain_logits = self.domain_classifier(self.grl(pooled_features))
        raw_domain_loss = F.cross_entropy(domain_logits, domain_labels)
        weighted_domain_loss = self.domain_loss_weight * raw_domain_loss

        self.last_domain_loss_raw = raw_domain_loss.detach()
        self.last_domain_accuracy = (
            (domain_logits.detach().argmax(dim=1) == domain_labels).float().mean()
        )
        return {**detection_losses, "loss_domain": weighted_domain_loss}


def build_base_detector():
    detector = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    )
    # Preserve the original mixed-species notebook's detection-head setup.
    in_features = detector.roi_heads.box_predictor.cls_score.in_features
    detector.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
    return detector


In [ ]:
def get_model():
    base_detector = build_base_detector()
    model = DomainAdaptiveMaskRCNN(
        detector=base_detector,
        num_domains=NUM_DOMAINS,
        domain_loss_weight=DOMAIN_LOSS_WEIGHT,
        grl_coefficient=GRL_COEFFICIENT,
    )
    return model.to(DEVICE)


In [ ]:
def save_checkpoint(state, filename):
    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)
    print(f"=> Validation Mask R-CNN task loss improved; saving {filename}")
    torch.save(state, filename)


In [ ]:
def load_checkpoint(checkpoint, model, optimizer, learning_rate):
    if checkpoint.get("domain_code_to_id") != DOMAIN_CODE_TO_ID:
        raise ValueError("Checkpoint domain mapping differs from this run.")
    if checkpoint.get("domain_loss_weight") != DOMAIN_LOSS_WEIGHT:
        raise ValueError("Checkpoint DOMAIN_LOSS_WEIGHT differs from this run.")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    for parameter_group in optimizer.param_groups:
        parameter_group["lr"] = learning_rate
    print(f"=> Loaded DA checkpoint from epoch {checkpoint.get('epoch', 'unknown')}")


## Preflight checks and training

Before full training, run the domain-label, GRL, loss-composition, and
gradient-connectivity checks below. Do not set `RUN_FULL_TRAINING=True`
until they all pass.


In [ ]:
train_dataset = CCDataset(mode="train", augmentation=transform)
valid_dataset = CCDataset(mode="valid")
test_dataset = CCDataset(mode="test")

for dataset, split_name in [
    (train_dataset, "train"),
    (valid_dataset, "valid"),
    (test_dataset, "test"),
]:
    validate_domain_coverage(dataset, split_name)

manifest_rows = []
for dataset, split_name in [
    (train_dataset, "train"),
    (valid_dataset, "valid"),
    (test_dataset, "test"),
]:
    manifest_rows.extend({"split": split_name, **sample} for sample in dataset.samples)
pd.DataFrame(manifest_rows).to_csv(OUTPUT_DIR / "domain_split_manifest.csv", index=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn,
)


In [ ]:
METRIC_NAMES = [
    "maskrcnn_loss",
    "domain_loss_raw",
    "domain_loss_weighted",
    "total_loss",
    "domain_accuracy",
]


def empty_metrics():
    return {name: 0.0 for name in METRIC_NAMES}


def loss_components(loss_dict, model):
    weighted_domain_loss = loss_dict["loss_domain"]
    maskrcnn_loss = sum(
        loss for name, loss in loss_dict.items() if name != "loss_domain"
    )
    total_loss = maskrcnn_loss + weighted_domain_loss
    return maskrcnn_loss, weighted_domain_loss, total_loss


def update_metrics(metric_sums, maskrcnn_loss, weighted_domain_loss, total_loss, model):
    metric_sums["maskrcnn_loss"] += maskrcnn_loss.detach().item()
    metric_sums["domain_loss_raw"] += model.last_domain_loss_raw.item()
    metric_sums["domain_loss_weighted"] += weighted_domain_loss.detach().item()
    metric_sums["total_loss"] += total_loss.detach().item()
    metric_sums["domain_accuracy"] += model.last_domain_accuracy.item()


def finalise_metrics(metric_sums, n_batches):
    if n_batches == 0:
        raise RuntimeError("No non-empty batches were processed.")
    return {name: value / n_batches for name, value in metric_sums.items()}


def train_one_epoch(loader, model, optimizer, device):
    model.train()
    metric_sums, n_batches = empty_metrics(), 0
    for batch in loader:
        if batch is None:
            continue
        images, targets, domain_labels = batch
        images = [image.to(device) for image in images]
        targets = [{key: value.to(device) for key, value in target.items()} for target in targets]
        domain_labels = domain_labels.to(device)

        loss_dict = model(images, targets, domain_labels)
        maskrcnn_loss, weighted_domain_loss, total_loss = loss_components(loss_dict, model)
        if not torch.isfinite(total_loss):
            raise FloatingPointError("Non-finite DA total loss encountered.")

        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        optimizer.step()

        update_metrics(
            metric_sums, maskrcnn_loss, weighted_domain_loss, total_loss, model
        )
        n_batches += 1
    return finalise_metrics(metric_sums, n_batches)


In [ ]:
best_validation_maskrcnn_loss = float("inf")


def validate(loader, model, optimizer, device, epoch):
    # Torchvision detection models return losses only in train mode.
    # Gradients are disabled, and no optimiser step is taken.
    global best_validation_maskrcnn_loss

    model.train()
    metric_sums, n_batches = empty_metrics(), 0

    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue

            images, targets, domain_labels = batch
            images = [image.to(device) for image in images]
            targets = [
                {key: value.to(device) for key, value in target.items()}
                for target in targets
            ]
            domain_labels = domain_labels.to(device)

            loss_dict = model(images, targets, domain_labels)
            maskrcnn_loss, weighted_domain_loss, total_loss = loss_components(
                loss_dict, model
            )

            if not torch.isfinite(total_loss):
                raise FloatingPointError(
                    "Non-finite validation DA total loss encountered."
                )

            update_metrics(
                metric_sums, maskrcnn_loss, weighted_domain_loss, total_loss, model
            )
            n_batches += 1

    metrics = finalise_metrics(metric_sums, n_batches)

    # MS・PSと統一：validation Mask R-CNN task lossが最小のepochを保存する
    if metrics["maskrcnn_loss"] < best_validation_maskrcnn_loss:
        best_validation_maskrcnn_loss = metrics["maskrcnn_loss"]

        if SAVE_MODEL:
            checkpoint = {
                "epoch": epoch,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "validation_metrics": metrics,
                "checkpoint_selection_metric": "valid_maskrcnn_loss",
                "domain_code_to_id": DOMAIN_CODE_TO_ID,
                "domain_loss_weight": DOMAIN_LOSS_WEIGHT,
                "grl_coefficient": GRL_COEFFICIENT,
                "random_seed": RANDOM_SEED,
                "torch_version": torch.__version__,
                "torchvision_version": torchvision.__version__,
            }
            save_checkpoint(checkpoint, CHECKPOINT_FILE)

    return metrics

In [ ]:
model = get_model()
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
print(model)


In [ ]:
if LOAD_MODEL:
    if not CHECKPOINT_FILE.exists():
        raise FileNotFoundError(CHECKPOINT_FILE)
    checkpoint = torch.load(CHECKPOINT_FILE, map_location=DEVICE)
    load_checkpoint(checkpoint, model, optimizer, LEARNING_RATE)


In [ ]:
def show_progress(frame, model, comment, history, keep_progress=False):
    model.eval()
    images = cv2.resize(frame, IMAGE_SIZE, cv2.INTER_LINEAR) / 255.0
    images = torch.as_tensor(images, dtype=torch.float32).unsqueeze(0)
    images = images.swapaxes(1, 3).swapaxes(2, 3)
    images = [image.to(DEVICE) for image in images]
    with torch.no_grad():
        prediction = model(images)

    cell_types = ["boundary", "vessel", "ray", "fiber", "parenchyma"]
    cell_colour = [(0, 0, 0), (1, 159 / 255, 155 / 255), (144 / 255, 229 / 255, 161 / 255),
                   (161 / 255, 201 / 255, 244 / 255), (1, 180 / 255, 130 / 255)]
    image_rgb = images[0].swapaxes(0, 2).swapaxes(0, 1).detach().cpu().numpy()
    overlay = np.zeros_like(image_rgb)
    for index in range(len(prediction[0]["masks"])):
        mask = prediction[0]["masks"][index, 0].detach().cpu().numpy()
        score = prediction[0]["scores"][index].item()
        cell_type = prediction[0]["labels"][index].item()
        if score > 0.6:
            overlay[mask > 0.67] = cell_colour[cell_type]

    progress = pd.DataFrame(history)
    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0, 0].imshow(cv2.addWeighted(image_rgb, 0.6, overlay, 0.4, 0))
    axes[0, 0].set_title(comment, fontsize=8)
    axes[0, 0].axis("off")
    axes[0, 1].plot(progress["epoch"], progress["train_maskrcnn_loss"], label="train")
    axes[0, 1].plot(progress["epoch"], progress["valid_maskrcnn_loss"], label="valid")
    axes[0, 1].set_title("Mask R-CNN task loss")
    axes[1, 0].plot(progress["epoch"], progress["train_domain_loss_raw"], label="train")
    axes[1, 0].plot(progress["epoch"], progress["valid_domain_loss_raw"], label="valid")
    axes[1, 0].set_title("Raw domain loss")
    axes[1, 1].plot(progress["epoch"], progress["train_total_loss"], label="train")
    axes[1, 1].plot(progress["epoch"], progress["valid_total_loss"], label="valid")
    axes[1, 1].set_title("Total loss")
    for axis in axes.flat[1:]:
        axis.set_xlabel("epoch")
        axis.legend()
    figure.tight_layout()
    if keep_progress:
        buffer = io.BytesIO()
        figure.savefig(buffer, format="png")
        plt.close(figure)
        return buffer.getvalue()
    plt.show()


In [ ]:
from IPython import display
import io


def first_nonempty_batch(loader):
    for batch in loader:
        if batch is not None:
            return batch
    raise RuntimeError("The loader contains no usable batches.")


def run_da_preflight_checks(model, loader, device, max_samples=2):
    '''Check labels, GRL sign, loss composition, and domain-gradient path.

    No optimiser step is taken, so this leaves the initial model weights unchanged.
    '''
    # Unit test: GRL must reverse an otherwise positive gradient exactly.
    probe = torch.ones((2, 3), requires_grad=True)
    GradientReversalLayer(1.0)(probe).sum().backward()
    assert torch.allclose(probe.grad, -torch.ones_like(probe)), "GRL sign check failed"

    images, targets, domain_labels = first_nonempty_batch(loader)
    images, targets, domain_labels = (
        images[:max_samples],
        targets[:max_samples],
        domain_labels[:max_samples],
    )
    assert len(images) == len(targets) == len(domain_labels)
    assert domain_labels.dtype == torch.long
    assert set(domain_labels.tolist()).issubset(set(DOMAIN_ID_TO_CODE))

    images = [image.to(device) for image in images]
    targets = [{key: value.to(device) for key, value in target.items()} for target in targets]
    domain_labels = domain_labels.to(device)
    model.train()
    model.zero_grad(set_to_none=True)
    loss_dict = model(images, targets, domain_labels)
    assert "loss_domain" in loss_dict
    assert model.last_domain_loss_raw is not None
    assert model.last_domain_accuracy is not None
    assert model.last_pooled_feature_shape[0] == len(images)

    maskrcnn_loss, weighted_domain_loss, total_loss = loss_components(loss_dict, model)
    assert torch.isfinite(maskrcnn_loss)
    assert torch.isfinite(weighted_domain_loss)
    assert torch.isfinite(total_loss)
    assert torch.allclose(total_loss, maskrcnn_loss + weighted_domain_loss)

    # The domain loss alone must reach both the domain head and the backbone.
    model.zero_grad(set_to_none=True)
    loss_dict["loss_domain"].backward()
    domain_head_has_gradient = any(
        parameter.grad is not None and torch.any(parameter.grad != 0)
        for parameter in model.domain_classifier.parameters()
    )
    backbone_has_gradient = any(
        parameter.grad is not None and torch.any(parameter.grad != 0)
        for parameter in model.detector.backbone.parameters()
    )
    assert domain_head_has_gradient, "Domain classifier received no gradient"
    assert backbone_has_gradient, "Domain loss is not connected to the backbone"
    model.zero_grad(set_to_none=True)

    report = {
        "n_images_checked": len(images),
        "domain_labels_checked": domain_labels.detach().cpu().tolist(),
        "maskrcnn_loss": maskrcnn_loss.detach().item(),
        "domain_loss_raw": model.last_domain_loss_raw.item(),
        "domain_loss_weighted": weighted_domain_loss.detach().item(),
        "total_loss": total_loss.detach().item(),
        "domain_accuracy": model.last_domain_accuracy.item(),
        "pooled_feature_shape": model.last_pooled_feature_shape,
        "grl_sign_check": "passed",
        "domain_gradient_path": "passed",
    }
    print("DA preflight checks passed:", report)
    return report


if RUN_SMOKE_TEST:
    preflight_report = run_da_preflight_checks(model, train_loader, DEVICE)
    pd.DataFrame([preflight_report]).to_csv(
        OUTPUT_DIR / "da_preflight_report.csv", index=False
    )


history = []
progress_frames = []
if RUN_FULL_TRAINING:
    for epoch in range(1, NUM_EPOCHS + 1):
        train_metrics = train_one_epoch(train_loader, model, optimizer, DEVICE)
        valid_metrics = validate(valid_loader, model, optimizer, DEVICE, epoch)
        record = {"epoch": epoch}
        record.update({f"train_{key}": value for key, value in train_metrics.items()})
        record.update({f"valid_{key}": value for key, value in valid_metrics.items()})
        history.append(record)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "epoch_metrics.csv", index=False)

        comment = (
            f"Epoch {epoch} | train total={train_metrics['total_loss']:.3f} | "
            f"valid total={valid_metrics['total_loss']:.3f}"
        )
        print(comment)
        display.clear_output(wait=True)
        frame = show_progress(img, model, comment, history, keep_progress=SAVE_PROGRESS_GIF)
        if SAVE_PROGRESS_GIF:
            progress_frames.append(frame)
else:
    print("Full training disabled. Review the preflight report, then set RUN_FULL_TRAINING=True.")


In [ ]:
if RUN_FULL_TRAINING:
    print("DA training finished.")


In [ ]:
import imageio

if RUN_FULL_TRAINING and SAVE_PROGRESS_GIF and progress_frames:
    imageio.mimsave(OUTPUT_DIR / "training_progress.gif", progress_frames, "GIF", fps=5)


In [ ]:
if history:
    metrics_df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(metrics_df["epoch"], metrics_df["train_maskrcnn_loss"], label="train")
    axes[0].plot(metrics_df["epoch"], metrics_df["valid_maskrcnn_loss"], label="valid")
    axes[0].set_title("Mask R-CNN task loss")
    axes[1].plot(metrics_df["epoch"], metrics_df["train_domain_loss_raw"], label="train")
    axes[1].plot(metrics_df["epoch"], metrics_df["valid_domain_loss_raw"], label="valid")
    axes[1].set_title("Raw domain loss")
    axes[2].plot(metrics_df["epoch"], metrics_df["train_domain_accuracy"], label="train")
    axes[2].plot(metrics_df["epoch"], metrics_df["valid_domain_accuracy"], label="valid")
    axes[2].axhline(1 / NUM_DOMAINS, linestyle="--", color="grey", label="chance")
    axes[2].set_title("Domain-classifier accuracy")
    for axis in axes:
        axis.set_xlabel("epoch")
        axis.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "learning_curves.png", dpi=300)
    plt.show()


# TEST

In [ ]:
import cv2
import io

In [ ]:

def predict_single_frame(frame):
    images = cv2.resize(frame, IMAGE_SIZE, cv2.INTER_LINEAR)/255
    images = torch.as_tensor(images, dtype=torch.float32).unsqueeze(0)
    images = images.swapaxes(1, 3).swapaxes(2, 3)
    images = list(image.to(DEVICE) for image in images)
    
    with torch.no_grad():
      pred = model(images)
    
    colors = [(0,0,0),(1,0,0),(0,1,0),(1,1,1),(0,0,1)]
    im = images[0].swapaxes(0, 2).swapaxes(0, 1).detach().cpu().numpy().astype(np.float32)
    im2 = np.zeros_like(im).astype(np.float32)
    
    msk=pred[0]['masks'].detach().cpu().numpy()
    scr=pred[0]['scores'].detach().cpu().numpy()
    box=pred[0]['boxes'].detach().cpu().numpy()
    cell_ids=pred[0]['labels'].detach().cpu().numpy()
    
    dfx = pd.DataFrame()
    dfx['labels'] = cell_ids
    dfx['scores'] = scr
    dfx['width'] = [x2-x1 for x1,y1,x2,y2 in list(box)]
    dfx['height'] = [y2-y1 for x1,y1,x2,y2 in list(box)]
    #area_w = [np.unique(k.flatten()>0.3, return_counts=True) for k in msk]
    #area = [j[1][1] for j in area_w]
    msk_thr = 0.67  # 可視化と揃えるなら 0.67 でもよい
    area = [int(np.count_nonzero(k > msk_thr)) for k in msk]
    dfx['area'] = area
    cell_types = ['boundary', 'vessel','ray','fiber', 'parenchyma']
    cell_type = [cell_types[n] for n in cell_ids]
    dfx['cell_name']= cell_type
    
    for i in range(len(pred[0]['masks'])):
        msk=pred[0]['masks'][i,0].detach().cpu().numpy()
        scr=pred[0]['scores'][i].detach().cpu().numpy()
        box=pred[0]['boxes'][i].detach().cpu().numpy()
        cell_name=pred[0]['labels'][i].detach().cpu().numpy()
        ####################
        
        if scr>0.05 :  #0.9
            #cv2.rectangle(im, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), colors[cell_type], 2)
            cv2.putText(im, str(cell_types[cell_name])+"_"+"{0:.2f}%".format(scr*100), (int(box[0]+5), int(box[1])+15), cv2.FONT_HERSHEY_SIMPLEX,
                        0.2, colors[cell_name], 1, cv2.LINE_AA)
            im2[:,:,0][msk> 0.37] = np.random.uniform(0,1)  # 0.87
            im2[:, :, 1][msk > 0.37] = np.random.uniform(0,1)
            im2[:, :, 2][msk > 0.37] = np.random.uniform(0,1)


    return (cv2.addWeighted(im, 0.5, im2, 0.5, 0)*255).astype(np.uint8),dfx, pred

def circle_plot(df):
    category_totals = df.groupby('cell_name')['area'].sum().reset_index()
    
    labels = category_totals['cell_name']
    sizes = category_totals['area']
    u, counts = np.unique(df['cell_name'], return_counts=True)
    colors = sns.color_palette('pastel')[0:len(labels)]
    
    items = [labels.values, labels]
    amount = [sizes.values, counts]
    comment = ['Occupied area of each cell','Number of each cell']
    
    fig, ax = plt.subplots(2,1,figsize=(4,4))
    for j in range(2):
        ax[j].pie(amount[j], labels=items[j], colors=colors, autopct='%1.1f%%', startangle=140)
        ax[j].axis('equal') 
        ax[j].set_title(comment[j])
    plt.tight_layout()
    plt.show()

def image_circle_plot(result,df, save_data=False):
    category_totals = df.groupby('cell_name')['area'].sum().reset_index()
    
    labels = category_totals['cell_name']
    sizes = category_totals['area']
    u, counts = np.unique(df['cell_name'], return_counts=True)
    colors = sns.color_palette('pastel')[0:len(labels)]
    
    items = [labels.values, labels]
    amount = [sizes.values, counts]
    comment = ['Occupied area of each cell','Number of each cell']

    plt.figure(figsize=(8,4))
    plt.subplot(121)
    plt.imshow(result)
    plt.axis('off')
    plt.subplot(222)
    plt.pie(amount[0], labels=items[0], colors=colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal') 
    plt.title(comment[0],fontsize=10)
    plt.subplot(224)
    plt.pie(amount[1], labels=items[1], colors=colors, autopct='%1.1f%%', startangle=140)
    plt.axis('equal') 
    plt.title(comment[1],fontsize=10)
    plt.tight_layout()
    if save_data == True:
        buf = io.BytesIO() # prepare buffer
        plt.savefig(buf, format='png') # store in the buffer
        enc = np.frombuffer(buf.getvalue(), dtype=np.uint8) # read from buffer
        dst = cv2.imdecode(enc, 1) # decoding
        dst = dst[:,:,::-1] # BGR->RGB
        plt.close()
        return dst
    else:
        plt.show()
    

In [ ]:
#from PIL import Image
#model.train(False)

#testfilename = [fl for fl in os.listdir(TEST_DIR) if fl.endswith('.png')]
#i = 4
#img0 = Image.open(os.path.join(TEST_DIR,testfilename[i])).convert('RGB')
#img0 = np.array(img0)
#img = cv2.imread(os.path.join(TEST_DIR,testfilename[i]))

#result, dfx, pred = predict_single_frame(img)


In [ ]:
#image_circle_plot(result, dfx)


In [ ]:
print("The end of script.")

The end of script.

an appendix for gifmovie making

'''
imx = Image.open('./dataset/original/Alllit_test_original.png').convert('RGB')

# moving range
theta = 360
rad = np.deg2rad(theta)
drad = rad/100
r = 400

images = []
count = 0
for i in tqdm(range(360)):
    count += 1
    rsinθ = r * np.sin(-drad*count)
    rcosθ = r * np.cos(-drad*count)
    x = rcosθ
    y = rsinθ
    imxp = imx.crop((1250+x, 1250+y, 1250+x+image_size, 1250+y+image_size))
    result_frame, df, _ = predict_single_frame(np.array(imxp))
    dst = image_circle_plot(result_frame, df, save_data=True)
    images.append(dst)

import imageio
imageio.mimsave('./output/result'+str(image_size)+'stat.gif', images,'GIF', fps=5)
'''

 show segmented and detected mask area 

result, _, pred = predict_single_frame(img)
plt.imshow(result)
plt.axis('off')
plt.show()


each = pred[0]['masks'][0].detach().cpu().numpy()[0]
each_mask = np.zeros_like(img0)
for i in range(3):
    each_mask[:,:,i] = each > 0.3
img_out = img0*each_mask
plt.imshow(img_out) 
plt.axis('off')
plt.show()

In [ ]:
#!ls ./checkpoint

In [ ]:
CHECKPOINT_FILE.exists()

In [ ]:
'''
!cp ./checkpoint/256_Alllit_777_epoch.tar ./model/256_Alllit_777_epoch.tar
!cp ./checkpoint/512_Alllit_777_epoch.tar ./model/512_Alllit_777_epoch.tar
'''

In [ ]:
#!cp ./checkpoint/256_Alllit_777_epoch.tar ./model/256_Alllit_777_epoch.tar

The end of script.